# Data Cleaning for a Social Media Ad Performance Dashboard

With the dataset [Social Media Advertisement Performance](https://www.kaggle.com/datasets/alperenmyung/social-media-advertisement-performance) from Kaggle, I need to clean the 4 separate tables to ensure clean analysis in the dashboard later.

In [1]:
import pandas as pd

### Import the Data

In [2]:
ad_events = pd.read_csv("../data/ad_events.csv")
ads = pd.read_csv("../data/ads.csv")
campaigns = pd.read_csv("../data/campaigns.csv")
users = pd.read_csv("../data/users.csv")

### Examine the different tables

In [3]:
ad_events.head()

,event_id,ad_id,user_id,timestamp,day_of_week,time_of_day,event_type
0,1,197,2359b,2025-07-26 00:19:56,Saturday,Night,Like
1,2,51,f9c67,2025-06-15 08:28:07,Sunday,Morning,Share
2,3,46,5b868,2025-06-27 00:40:02,Friday,Night,Impression
3,4,166,3d440,2025-06-05 19:20:45,Thursday,Evening,Impression
4,5,52,68f1a,2025-07-22 08:30:29,Tuesday,Morning,Impression


In [4]:
ads.head()

,ad_id,campaign_id,ad_platform,ad_type,target_gender,target_age_group,target_interests
0,1,28,Facebook,Video,Female,35-44,"art, technology"
1,2,33,Facebook,Stories,All,25-34,"travel, photography"
2,3,20,Instagram,Carousel,All,25-34,technology
3,4,28,Facebook,Stories,Female,25-34,news
4,5,24,Instagram,Image,Female,25-34,news


In [5]:
campaigns.head()

,campaign_id,name,start_date,end_date,duration_days,total_budget
0,1,Campaign_1_Launch,2025-05-25,2025-07-23,59,24021.32
1,2,Campaign_2_Launch,2025-04-16,2025-07-07,82,79342.41
2,3,Campaign_3_Winter,2025-05-04,2025-06-29,56,14343.25
3,4,Campaign_4_Summer,2025-06-04,2025-08-08,65,45326.60
4,5,Campaign_5_Launch,2025-07-11,2025-08-28,48,68376.69


In [6]:
users.head()

,user_id,user_gender,user_age,age_group,country,location,interests
0,a2474,Female,24,18-24,United Kingdom,New Mariomouth,"fitness, health"
1,141e5,Male,21,18-24,Germany,Danielsfort,"food, fitness, lifestyle"
2,34db0,Male,27,25-34,Australia,Vincentchester,"fashion, news"
3,20d08,Female,28,25-34,India,Lisaport,"health, news, finance"
4,9e830,Male,28,25-34,United States,Brownmouth,"health, photography, lifestyle"


### Table Shape, Data Type Check, & Null Check

In [7]:
for dfname, df in [('ad_events', ad_events), ('ads', ads), ('campaigns', campaigns), ('users', users)]:
    print(f"-- {dfname} --")
    print(f'shape: {df.shape}')
    print(df.dtypes)
    print("nulls:")
    print(df.isnull().sum()[df.isnull().sum() > 0])
    print()

-- ad_events --
shape: (400000, 7)
event_id       int64
ad_id          int64
user_id          str
timestamp        str
day_of_week      str
time_of_day      str
event_type       str
dtype: object
nulls:
Series([], dtype: int64)

-- ads --
shape: (200, 7)
ad_id               int64
campaign_id         int64
ad_platform           str
ad_type               str
target_gender         str
target_age_group      str
target_interests      str
dtype: object
nulls:
Series([], dtype: int64)

-- campaigns --
shape: (50, 6)
campaign_id        int64
name                 str
start_date           str
end_date             str
duration_days      int64
total_budget     float64
dtype: object
nulls:
Series([], dtype: int64)

-- users --
shape: (10000, 7)
user_id          str
user_gender      str
user_age       int64
age_group        str
country          str
location         str
interests        str
dtype: object
nulls:
Series([], dtype: int64)



Besides the date fields in the campaigns table which I will take care of later, all data types are what they should be and no apparent null values.

### Revenue Impact

The initial idea is to measure the ad campaign success by revenue generated. But, across the 4 tables I don't see a revenue column. It could be hiding in the `event_type` column. 

In [8]:
ad_events['event_type'].value_counts()

event_type
Impression    339812
Click          40079
Like           12013
Comment         4108
Purchase        2031
Share           1957
Name: count, dtype: int64

No revenue value info available. With that lack of info, I will have to measure campaign success by a conversion funnel. Thankfully we do have the users events tracked, so we can pivot to campaign success measuring the efficiency of each ads impact by the funnel.

### Reference Key Check

In [9]:
# key check: ad_events.ad_id AND ads.ad_id
lost_ads = set(ad_events['ad_id']) - set(ads['ad_id'])
print(f"ad_events with no matching ad: {len(lost_ads)}")

# key check: ad_events.user_id AND users.user_id
lost_users = set(ad_events['user_id']) - set(users['user_id'])
print(f"ad_events with no matching user: {len(lost_users)}")

# key check: ads.campaign_id AND campaigns.campaign_id
lost_campaigns = set(ads['campaign_id']) - set(campaigns['campaign_id'])
print(f"ads with no matching campaign: {len(lost_campaigns)}")

ad_events with no matching ad: 0
ad_events with no matching user: 0
ads with no matching campaign: 0


### Checking Duplicates

In [10]:
print(f"duplicates in ad_events: {ad_events['event_id'].duplicated().sum()}")
print(f"duplicates in ads: {ads['ad_id'].duplicated().sum()}")
print(f"duplicates in campaigns: {campaigns['campaign_id'].duplicated().sum()}")
print(f"duplicates in users: {users['user_id'].duplicated().sum()}")

duplicates in ad_events: 0
duplicates in ads: 0
duplicates in campaigns: 0
duplicates in users: 50


With 50 duplicates in users, I am simply going to delete those duplicate `user_id` rows.

In [11]:
print(f"user rows before dropping duplicates: {len(users)}")
users = users.drop_duplicates(subset = ['user_id'])
print(f"user rows after dropping duplicates: {len(users)}")

user rows before dropping duplicates: 10000
user rows after dropping duplicates: 9950


### Parse the Date/Datetime Columns

As mentioned previously, I would like to adjust date fields in the campaign table as they were loaded as strings by default. I am going to convert them as I want to use them as datetime for future analysis.

In [12]:
ad_events['timestamp'] = pd.to_datetime(ad_events['timestamp'])
campaigns['start_date'] = pd.to_datetime(campaigns['start_date'])
campaigns['end_date'] = pd.to_datetime(campaigns['end_date'])

print(ad_events['timestamp'].dtype, campaigns['start_date'].dtype, campaigns['end_date'].dtype)

datetime64[us] datetime64[us] datetime64[us]


### Campaign Date Check

With this imported data, I want to ensure there are no issues regarding campaign dates and the events coinciding with those campaigns. If there is an issue it is worth noting and pivoting in the later analysis and dashboard creation.

In [13]:
print(f"first ad event: {ad_events['timestamp'].min()}")
print(f"last ad event: {ad_events['timestamp'].max()}")

first ad event: 2025-05-07 14:11:57
last ad event: 2025-08-06 14:11:30


In [14]:
events_w_campaign = ad_events.merge(ads[['ad_id', 'campaign_id']], on='ad_id', how='left') \
                                  .merge(campaigns[['campaign_id', 'start_date', 'end_date']], on='campaign_id', how='left')

out_of_range = events_w_campaign[
    (events_w_campaign['timestamp'] < events_w_campaign['start_date']) |
    (events_w_campaign['timestamp'] > events_w_campaign['end_date'])
]
print(f"events outside the campaign's date range: {len(out_of_range)} ({len(out_of_range)/len(events_w_campaign):.2%})")

events outside the campaign's date range: 225406 (56.35%)


56% is a significant amount of events happening outside the campaigns flight dates. That number is too large to simply filter out without having an understanding why. Do these events happen before the campaigns start, or after it ends, or both?

In [15]:
before_start = out_of_range[out_of_range['timestamp'] < out_of_range['start_date']]
after_end = out_of_range[out_of_range['timestamp'] > out_of_range['end_date']]

print(f"before campaign: {len(before_start)} ({len(before_start)/len(events_w_campaign):.2%})")
print(f"after campaign: {len(after_end)} ({len(after_end)/len(events_w_campaign):.2%})")

before campaign: 66578 (16.64%)
after campaign: 158828 (39.71%)


Events are happening both before and after campaign date range. Are the events significantly outside the campaign date range, or by just a little?

In [16]:
before_start_days = (before_start['start_date'] - before_start['timestamp']).dt.days
after_end_days = (after_end['timestamp'] - after_end['end_date']).dt.days

print("days before start_date:")
print(before_start_days.describe())
print()
print("days after end_date:")
print(after_end_days.describe())

days before start_date:
count    66578.000000
mean        24.050873
std         18.436703
min          0.000000
25%          8.000000
50%         20.000000
75%         37.000000
max         76.000000
dtype: float64

days after end_date:
count    158828.000000
mean         37.056413
std          27.611241
min           0.000000
25%          14.000000
50%          32.000000
75%          55.000000
max         126.000000
dtype: float64


Both ends of the issue are significantly outside the date range. This could simply be an artifact issue with the synthetic data.

In [17]:
print(f"ad_events timestamp range: {ad_events['timestamp'].min()} to {ad_events['timestamp'].max()}")
print(f"campaigns start_date range: {campaigns['start_date'].min()} to {campaigns['start_date'].max()}")
print(f"campaigns end_date range:   {campaigns['end_date'].min()} to {campaigns['end_date'].max()}")

ad_events timestamp range: 2025-05-07 14:11:57 to 2025-08-06 14:11:30
campaigns start_date range: 2025-02-13 00:00:00 to 2025-07-23 00:00:00
campaigns end_date range:   2025-04-02 00:00:00 to 2025-10-12 00:00:00


It looks like the event timestamps weren't constrained to campaign flight dates. It looks be an artifact rather than a real indicator of delayed engagment or issues with the ads themselves. With that, I will attribute events by ID relationship instead of date filtering.

### Building the Events Table

For the exploratory dashboard itself, I want to sync all the data together to easily summarize the key components for decision makers.

In [18]:
events_master = (
    ad_events
    .merge(ads, on='ad_id', how='left')
    .merge(campaigns[['campaign_id', 'name', 'start_date', 'end_date', 'duration_days', 'total_budget']],
           on='campaign_id', how='left')
    .merge(users, on='user_id', how='left', suffixes=('', '_user'))
)

print(events_master.shape)
events_master.head()

(400000, 24)


,event_id,ad_id,user_id,timestamp,day_of_week,time_of_day,event_type,campaign_id,ad_platform,ad_type,...,start_date,end_date,duration_days,total_budget,user_gender,user_age,age_group,country,location,interests
0,1,197,2359b,2025-07-26 00:19:56,Saturday,Night,Like,9,Facebook,Stories,...,2025-05-25,2025-07-13,49,40094.07,Female,24,18-24,United States,West Shawna,"gaming, food"
1,2,51,f9c67,2025-06-15 08:28:07,Sunday,Morning,Share,26,Instagram,Carousel,...,2025-04-01,2025-06-17,77,44538.87,Female,30,25-34,United States,Meyersland,"photography, finance"
2,3,46,5b868,2025-06-27 00:40:02,Friday,Night,Impression,10,Instagram,Carousel,...,2025-05-17,2025-07-21,65,19669.27,Male,20,18-24,United States,Barrerahaven,"fashion, sports, travel"
3,4,166,3d440,2025-06-05 19:20:45,Thursday,Evening,Impression,14,Instagram,Image,...,2025-04-15,2025-06-04,50,39849.94,Female,18,18-24,United States,Lake Angelaland,"food, art"
4,5,52,68f1a,2025-07-22 08:30:29,Tuesday,Morning,Impression,2,Instagram,Stories,...,2025-04-16,2025-07-07,82,79342.41,Male,58,55-65,United Kingdom,Robinsonberg,"finance, lifestyle"


### Null Check after Merge

In [19]:
events_master.isnull().sum()

event_id            0
ad_id               0
user_id             0
timestamp           0
day_of_week         0
time_of_day         0
event_type          0
campaign_id         0
ad_platform         0
ad_type             0
target_gender       0
target_age_group    0
target_interests    0
name                0
start_date          0
end_date            0
duration_days       0
total_budget        0
user_gender         0
user_age            0
age_group           0
country             0
location            0
interests           0
dtype: int64

### Building the Campaigns Summary Table

One row per campaign, with funnel counts and four metrics:
- **CTR** — clicks / impressions
- **Engagement rate** — likes + shares + comments / impressions
- **Conversion rate** — purchases / clicks
- **Cost per purchase** — total campaign budget / purchases (an estimate since budget isn't tracked per event, just at the campaign level)

In [20]:
campaign_summary = (
    events_master
    .pivot_table(index=['campaign_id', 'name', 'total_budget', 'duration_days'],
                 columns='event_type', values='event_id', aggfunc='count', fill_value=0)
    .reset_index()
)
campaign_summary.columns.name = None

campaign_summary['ctr'] = campaign_summary['Click'] / campaign_summary['Impression'].replace(0, pd.NA)
campaign_summary['engagement_rate'] = (campaign_summary['Like'] + campaign_summary['Share'] + campaign_summary['Comment']) / campaign_summary['Impression'].replace(0, pd.NA)
campaign_summary['conversion_rate'] = campaign_summary['Purchase'] / campaign_summary['Click'].replace(0, pd.NA)
campaign_summary['cost_per_purchase'] = campaign_summary['total_budget'] / campaign_summary['Purchase'].replace(0, pd.NA)

campaign_summary.head()

,campaign_id,name,total_budget,duration_days,Click,Comment,Impression,Like,Purchase,Share,ctr,engagement_rate,conversion_rate,cost_per_purchase
0,1,Campaign_1_Launch,24021.32,59,578,50,5045,188,33,25,0.114569,0.052131,0.057093,727.918788
1,2,Campaign_2_Launch,79342.41,82,599,64,5255,172,31,25,0.113987,0.049667,0.051753,2559.432581
2,3,Campaign_3_Winter,14343.25,56,783,79,6711,240,41,41,0.116674,0.053643,0.052363,349.835366
3,4,Campaign_4_Summer,45326.60,65,1178,121,10249,356,49,74,0.114938,0.053761,0.041596,925.032653
4,5,Campaign_5_Launch,68376.69,48,583,69,5101,218,34,28,0.114291,0.061753,0.058319,2011.079118


In [21]:
campaign_summary.isnull().sum()

campaign_id          0
name                 0
total_budget         0
duration_days        0
Click                0
Comment              0
Impression           0
Like                 0
Purchase             0
Share                0
ctr                  0
engagement_rate      0
conversion_rate      0
cost_per_purchase    0
dtype: int64

### Platform Summary Table

Same funnel counts and metrics as above, rolled up by `ad_platform` instead of campaign.

In [22]:
platform_summary = (
    events_master
    .pivot_table(index='ad_platform', columns='event_type', values='event_id', aggfunc='count', fill_value=0)
    .reset_index()
)
platform_summary.columns.name = None

platform_summary['ctr'] = platform_summary['Click'] / platform_summary['Impression'].replace(0, pd.NA)
platform_summary['engagement_rate'] = (platform_summary['Like'] + platform_summary['Share'] + platform_summary['Comment']) / platform_summary['Impression'].replace(0, pd.NA)
platform_summary['conversion_rate'] = platform_summary['Purchase'] / platform_summary['Click'].replace(0, pd.NA)

platform_summary

,ad_platform,Click,Comment,Impression,Like,Purchase,Share,ctr,engagement_rate,conversion_rate
0,Facebook,25389,2632,215972,7505,1323,1275,0.117557,0.052840,0.052109
1,Instagram,14690,1476,123840,4508,708,682,0.118621,0.053828,0.048196


In [23]:
platform_summary.isnull().sum()

ad_platform        0
Click              0
Comment            0
Impression         0
Like               0
Purchase           0
Share              0
ctr                0
engagement_rate    0
conversion_rate    0
dtype: int64

### Export

In [25]:
events_master.to_csv('../data/events_master.csv', index=False)
campaign_summary.to_csv('../data/campaign_summary.csv', index=False)
platform_summary.to_csv('../data/platform_summary.csv', index=False)

### Summary

Taking the raw event, ad, campaign, and user tables and converting them into analysis ready outputs: `events_master.csv` (full event level detail, used for the exploratory parts of the dashboard) and `campaign_summary.csv` / `platform_summary.csv` (pre-aggregated rollups). I found and handled 50 duplicate user records, confirmed there were no orphaned foreign keys, and identified the event timestamps aren't constrained to campaign flight dates, so I instead handled it by attributing events to campaigns by ID.